In [52]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os

# --- Ajuste de sys.path para tu módulo RRAM ---
base_path = Path("c:/Users/Usuario/Documents/GitHub/RRAM_Simulation")
if str(base_path) not in sys.path:
    sys.path.append(str(base_path))

# Ahora importa el módulo
from RRAM.Representate import config_ax, setup_plt
from RRAM import io_manager as io 

# --- Configuración de la figura ---
# Inicializa el estilo global
setup_plt(plt, latex=True, scaling=2)

In [53]:
# --- Carpeta destino de todas las figuras ---
figures_dir = base_path / "Results" / "Media_set"
figures_dir.mkdir(parents=True, exist_ok=True)

# --- Listar carpetas de simulación bajo Results/ ---
results_dir = base_path / "Results"
sim_dirs = sorted(
    d for d in results_dir.iterdir()
    if d.is_dir() and d.name.startswith("simulation_")
)
print(f"🔄 Encontradas {len(sim_dirs)} simulaciones: {[d.name for d in sim_dirs]}")


🔄 Encontradas 5 simulaciones: ['simulation_1', 'simulation_2', 'simulation_3', 'simulation_4', 'simulation_5']


In [54]:
# Diccionario para almacenar la columna de Tiempo y las columnas de Intensidad de cada simulación
all_data_for_df_set = {}
all_data_for_df_reset = {}

for sim_dir in sim_dirs:
    num = sim_dir.name.split("_")[1]
    print(f"\n Procesando simulación {num} Ruta: {sim_dir}")
    # 1) Construir rutas de datos
    paths = {
        "csv_set": sim_dir / "set" / f"Resultados_set_{num}.csv",
        "csv_reset": sim_dir / "reset" / f"Resultados_reset_{num}.csv",
    }
    missing = [k for k, p in paths.items() if not p.exists()]
    if missing:
        print(f"⚠️ Sim {num}: faltan {missing}, omitiendo.")
        continue

    # 2) Cargar datos
    data_set = io.leer_csv(str(paths["csv_set"]))
    data_reset = io.leer_csv(str(paths["csv_reset"]))

    
    # La añadimos solo una vez el voltaje, preferiblemente en la primera iteración o antes.
    if "Voltaje [V]" not in all_data_for_df_set and "Voltaje [V]" not in all_data_for_df_reset:
        all_data_for_df_set["Voltaje [V]"] = data_set["Voltaje [V]"]
        all_data_for_df_reset["Voltaje [V]"] = data_reset["Voltaje [V]"]
  
    # Añadimos la columna de intensidad de la simulación actual con un nombre único
    all_data_for_df_set[f"Intensidad_Sim_{num} [A]"] = data_set["Intensidad [A]"]
    
    # Añadimos la columna de intensidad de la simulación actual con un nombre único
    all_data_for_df_reset[f"Intensidad_Sim_{num} [A]"] = data_reset["Intensidad [A]"]

# Convertir los diccionarios a DataFrames
df_set = pd.DataFrame(all_data_for_df_set)
df_reset = pd.DataFrame(all_data_for_df_reset)


 Procesando simulación 1 Ruta: c:\Users\Usuario\Documents\GitHub\RRAM_Simulation\Results\simulation_1

 Procesando simulación 2 Ruta: c:\Users\Usuario\Documents\GitHub\RRAM_Simulation\Results\simulation_2

 Procesando simulación 3 Ruta: c:\Users\Usuario\Documents\GitHub\RRAM_Simulation\Results\simulation_3

 Procesando simulación 4 Ruta: c:\Users\Usuario\Documents\GitHub\RRAM_Simulation\Results\simulation_4

 Procesando simulación 5 Ruta: c:\Users\Usuario\Documents\GitHub\RRAM_Simulation\Results\simulation_5


In [55]:
ruta_archivo_set = os.getcwd() + '/Datos_Experimentales/Ciclos_Experimentales/Cycle_p_1000.txt'
ruta_archivo_reset = os.getcwd() + '/Datos_Experimentales/Ciclos_Experimentales/Cycle_n_1000.txt'

# Cargar datos experimentales
data_set = np.loadtxt(ruta_archivo_set)
data_reset = np.loadtxt(ruta_archivo_reset)

x_set = data_set[:, 0]
y_set = data_set[:, 1]

x_reset = data_reset[:, 0]
y_reset = abs(data_reset[:, 1])

V_medidas = np.concatenate((x_set, x_reset))
I_medidas = np.concatenate((y_set, y_reset))

In [56]:
# 1. Identificar las columnas de intensidad usando el prefijo común
intensity_columns = [col for col in df_set.columns if col.startswith("Intensidad_Sim_")]

print(f"🔍 Columnas de Intensidad encontradas: {intensity_columns}")
factors_reset = [0.9, 0.95, 1, 1.05, 1.1]
E_r_base = 1.01
E_r = [round(E_r_base * f, 3) for f in factors_reset]

fig, axes = plt.subplots(figsize=(12, 9))

config_ax(axes)

# Configurar etiquetas y título
axes.set_xlabel(r"Voltage (\si{\volt})")  # (\si{\nano\meter^{-1}})
axes.set_ylabel(r"Intensity (\si{\ampere})")
axes.set_title(r"Intensidad Media durante el set a distintas $E_r$", pad=20)
plt.legend()  # Muestra la leyenda con los nombres de las series

# Obtener los colores predeterminados de matplotlib
colors = [
    '#FF6B6B',  # Coral oscuro
    '#4DC786',  # Verde bosque
    '#5B8FF9',  # Azul real
    '#EFAA5C',  # Naranja tostado
    '#8B72BE'   # Púrpura medio
]
k = 0
for col in intensity_columns:
    current_color = colors[k % len(colors)]  # Usar módulo para no exceder la lista de colores

    # # SET con color actual
    # plt.scatter(df_set["Voltaje [V]"], df_set[col],
    #             label=rf'$E_r = {E_r[k]}$',
    #             alpha=0.6, s=3,
    #             color=current_color)

    # RESET con el mismo color
    plt.scatter(df_reset["Voltaje [V]"], df_reset[col],
                label=rf'$E_r = {E_r[k]}$',
                alpha=0.6, s=1,
                color=current_color)
    k += 1

# axes.plot(x_set, y_set,
#           label="Intensidad experimental",
#           color="black", linewidth=2)  # Línea roja y más gruesa

axes.plot(x_reset, y_reset,
          label="Intensidad experimental",
          color="black", linewidth=2)  # Línea roja y más gruesa
plt.yscale('log')

axes.legend(
    markerscale=4,          # Tamaño del marcador
    fontsize='medium',      # Tamaño del texto
    loc='best'             # Ubicación automática
)

plt.savefig(figures_dir / f"I-V_varias_E_r_reset.png", dpi=300, bbox_inches='tight')
plt.close(fig)

🔍 Columnas de Intensidad encontradas: ['Intensidad_Sim_1 [A]', 'Intensidad_Sim_2 [A]', 'Intensidad_Sim_3 [A]', 'Intensidad_Sim_4 [A]', 'Intensidad_Sim_5 [A]']


C:\Users\Usuario\AppData\Local\Temp\ipykernel_13876\2461310048.py:17: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()  # Muestra la leyenda con los nombres de las series
